# Compare different Berkeley Earth realisations

In [ ]:
import matplotlib.pyplot as plt
import xarray as xr

from utils import berkeley

## Config


In [ ]:
old = "v2024"
new = "v2025"

## Difference in global mean

In [ ]:
berkeley_globmean_1850_1900_old = berkeley.read_globmean(
    ref_period=slice(1850, 1900), version=old
)
berkeley_globmean_1850_1900_new = berkeley.read_globmean(
    ref_period=slice(1850, 1900), version=new
)

In [ ]:
berkeley_globmean_1850_1900_old.plot()
berkeley_globmean_1850_1900_new.plot()

In [ ]:
berkeley_globmean_abs_old = berkeley.read_globmean(ref_period=None, version=old)
berkeley_globmean_abs_new = berkeley.read_globmean(ref_period=None, version=new)

In [ ]:
berkeley_globmean_abs_old.plot()
berkeley_globmean_abs_new.plot()

In [ ]:
diff_anom = berkeley_globmean_1850_1900_old - berkeley_globmean_1850_1900_new

diff_abs = berkeley_globmean_abs_old - berkeley_globmean_abs_new

f, ax = plt.subplots()

diff_anom.plot(ax=ax, label="v2024 - v2025: anom. temp.")
diff_abs.plot(ax=ax, label="v2024 - v2025: abs. temp.")

plt.legend()

ax.axhline(0, lw=0.5, c="0.1")

## Difference in spatial data

In [ ]:
period = slice("1955", "1960")

tmax_new = berkeley.read_full("TMAX", version=new)
tmin_new = berkeley.read_full("TMIN", version=new)

In [ ]:
tmax_old = berkeley.read_full("TMAX", version=old)
tmin_old = berkeley.read_full("TMIN", version=old)

In [ ]:
tmax_old.climatology.load()
tmax_new.climatology.load()
None

In [ ]:
tmin_old.climatology.load()
tmin_new.climatology.load()
None

In [ ]:
tmax_old

In [ ]:
tmax_new

### Difference in climatology

In [ ]:
f, ax = plt.subplots()

h = (
    (tmax_new.climatology - tmax_old.climatology)
    .mean("day_number")
    .plot(ax=ax, add_colorbar=False)
)

cbar = plt.colorbar(h, ax=ax)

cbar.set_label("climatology difference (°C)")

ax.set_title(
    "Annual mean difference in TMAX climatology (v2025 - v2024)", loc="left", fontsize=8
)

# plt.savefig("BerkeleyEarth_TMAX_climatology_diff.png")

In [ ]:
(tmin_old.climatology - tmin_new.climatology).mean("day_number").plot()

In [ ]:
lon = 8
lat = 47

# (tmax_old.climatology - tmax_new.climatology).sel(lat=lat, lon=lon, method="nearest").plot()

f, axs = plt.subplots(1, 2, sharey=True)


ax = axs[0]
tmin_old.climatology.sel(lat=lat, lon=lon, method="nearest").plot(
    ax=ax, label=old, color="r"
)
tmin_new.climatology.sel(lat=lat, lon=lon, method="nearest").plot(
    ax=ax, label=new, color="b"
)
ax.set_title("TMIN")
ax.set_ylabel("Temperature (°C)")
ax.legend()

ax = axs[1]
tmax_old.climatology.sel(lat=lat, lon=lon, method="nearest").plot(
    ax=ax, label=old, color="r"
)
tmax_new.climatology.sel(lat=lat, lon=lon, method="nearest").plot(
    ax=ax, label=new, color="b"
)
ax.set_title("TMAX")
ax.set_ylabel("")
ax.legend()

In [ ]:
a = tmin_old.climatology.sel(lat=lat, lon=lon, method="nearest")
b = tmax_old.climatology.sel(lat=lat, lon=lon, method="nearest")

plt.fill_between(a.day_number, a, b, color="#3182bd88", label=old)


a = tmin_new.climatology.sel(lat=lat, lon=lon, method="nearest")
b = tmax_new.climatology.sel(lat=lat, lon=lon, method="nearest")

plt.fill_between(a.day_number, a, b, color="#c51b8a88", label=new)

plt.legend()

plt.title("TX and TN difference")

### Compare TXx

In [ ]:
tmax_seas_old = berkeley.add_climatology(tmax_old)

In [ ]:
tmax_seas_new = berkeley.add_climatology(tmax_new)

In [ ]:
with xr.set_options(use_flox=True):
    txx_old = tmax_seas_old.groupby("time.year").max("time", engine="flox").compute()

In [ ]:
with xr.set_options(use_flox=True):
    txx_new = tmax_seas_new.groupby("time.year").max("time", engine="flox").compute()

In [ ]:
(txx_new - txx_old).sel(year=2000).plot(robust=True)